In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

# โหลดข้อมูลจากโฟลเดอร์ dataset
file_name = "dataset - 2020-09-24.csv"
path_candidates = [
    Path.cwd() / file_name,
    Path.cwd().parent / file_name,
    Path.cwd().parent / "dataset" / file_name,
]
DATA_PATH = next((path for path in path_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"ไม่พบไฟล์ {file_name} ในตำแหน่ง {path_candidates}")
dataset = pd.read_csv(DATA_PATH)

# เตรียมข้อมูลเหมือนกันสำหรับทั้ง ANN และ SVM
percent_columns = [
    "Tackle success %", "Shooting accuracy %", "Cross accuracy %"
]
for column in percent_columns:
    dataset[column] = pd.to_numeric(
        dataset[column].astype("string").str.rstrip("%"), errors="coerce"
    ) / 100

categorical_columns = ["Club", "Position", "Nationality"]
for column in categorical_columns:
    dataset[column] = dataset[column].astype("string").fillna("Unknown")

for column in dataset.columns:
    if column not in categorical_columns + ["Name"] + percent_columns:
        dataset[column] = pd.to_numeric(dataset[column], errors="coerce")

# ทำนาย Goals และตัดข้อมูลที่เป็น target leakage
TARGET_COLUMN = "Goals"
leakage_columns = [
    "Goals", "Goals per match", "Headed goals", "Goals with right foot",
    "Goals with left foot", "Own goals"
]
feature_columns = [
    column for column in dataset.columns
    if column not in leakage_columns + ["Name"]
]
model_data = dataset[feature_columns + [TARGET_COLUMN]].dropna(subset=[TARGET_COLUMN])
X = model_data[feature_columns]
y = model_data[TARGET_COLUMN]

# ใช้ชุด train/test เดียวกันเพื่อเปรียบเทียบอย่างยุติธรรม
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [
    column for column in X.columns if column not in numeric_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical_features,
        ),
    ],
    remainder="drop",
)

models = {
    "ANN": MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=40,
        random_state=42,
    ),
    "SVM": SVR(
        kernel="rbf",
        C=10.0,
        epsilon=0.1,
        gamma="scale",
    ),
}

trained_models = {}
comparison_rows = []
prediction_table = X_test[["Club", "Position"]].copy()
prediction_table["ประตูจริง"] = y_test

for model_name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    predictions = np.maximum(pipeline.predict(X_test), 0)
    trained_models[model_name] = pipeline
    comparison_rows.append({
        "โมเดล": model_name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions),
    })
    prediction_table[f"{model_name} ทำนาย"] = predictions.round(2)

comparison = pd.DataFrame(comparison_rows).sort_values("RMSE").reset_index(drop=True)
best_model = comparison.loc[0, "โมเดล"]
ann_result = comparison.loc[comparison["โมเดล"] == "ANN"].iloc[0]
svm_result = comparison.loc[comparison["โมเดล"] == "SVM"].iloc[0]

print("การเปรียบเทียบ ANN และ SVM")
print(f"ไฟล์ข้อมูล: {DATA_PATH}")
print(f"จำนวนข้อมูล: {len(model_data):,} แถว")
print(f"Train: {len(X_train):,} แถว | Test: {len(X_test):,} แถว")
print("\nผลการประเมิน:")
print(comparison.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print(f"\nโมเดลที่ดีที่สุดจาก RMSE: {best_model}")
print("\nเหตุผลที่เลือกโมเดลนี้:")
print("- RMSE และ MAE ยิ่งต่ำ แปลว่าค่าทำนายคลาดเคลื่อนน้อย")
print("- R² ยิ่งสูง แปลว่าโมเดลอธิบายความแปรปรวนของข้อมูลได้ดี")
print(
    f"- ANN มี RMSE {ann_result['RMSE']:.4f} เทียบกับ SVM {svm_result['RMSE']:.4f} "
    f"จึงคลาดเคลื่อนน้อยกว่า {svm_result['RMSE'] - ann_result['RMSE']:.4f}"
)
print(
    f"- ANN มี MAE {ann_result['MAE']:.4f} เทียบกับ SVM {svm_result['MAE']:.4f} "
    f"จึงมีความคลาดเคลื่อนเฉลี่ยต่ำกว่า {svm_result['MAE'] - ann_result['MAE']:.4f}"
)
print(
    f"- ANN มี R² {ann_result['R2']:.4f} เทียบกับ SVM {svm_result['R2']:.4f} "
    f"จึงอธิบายข้อมูลได้ดีกว่า {ann_result['R2'] - svm_result['R2']:.4f}"
)
print(f"สรุป: เลือกใช้ {best_model} เพราะให้ผลดีที่สุดบนชุดทดสอบเดียวกันทั้งสามตัวชี้วัด")
print("\nตัวอย่างผลการทำนาย:")
print(prediction_table.head(10).to_string())

การเปรียบเทียบ ANN และ SVM
ไฟล์ข้อมูล: c:\Users\guyza\OneDrive\Desktop\PML\PML\dataset\dataset - 2020-09-24.csv
จำนวนข้อมูล: 571 แถว
Train: 456 แถว | Test: 115 แถว

ผลการประเมิน:
โมเดล    MAE   RMSE     R2
  ANN 1.8126 4.1524 0.8842
  SVM 1.8840 4.5748 0.8595

โมเดลที่ดีที่สุดจาก RMSE: ANN

เหตุผลที่เลือกโมเดลนี้:
- RMSE และ MAE ยิ่งต่ำ แปลว่าค่าทำนายคลาดเคลื่อนน้อย
- R² ยิ่งสูง แปลว่าโมเดลอธิบายความแปรปรวนของข้อมูลได้ดี
- ANN มี RMSE 4.1524 เทียบกับ SVM 4.5748 จึงคลาดเคลื่อนน้อยกว่า 0.4224
- ANN มี MAE 1.8126 เทียบกับ SVM 1.8840 จึงมีความคลาดเคลื่อนเฉลี่ยต่ำกว่า 0.0714
- ANN มี R² 0.8842 เทียบกับ SVM 0.8595 จึงอธิบายข้อมูลได้ดีกว่า 0.0248
สรุป: เลือกใช้ ANN เพราะให้ผลดีที่สุดบนชุดทดสอบเดียวกันทั้งสามตัวชี้วัด

ตัวอย่างผลการทำนาย:
                         Club    Position  ประตูจริง  ANN ทำนาย  SVM ทำนาย
509      West-Bromwich-Albion  Midfielder          0       0.00       0.00
70   Brighton-and-Hove-Albion    Defender          0       0.00       0.00
131                   Chelsea  Mid